## A Geo Enabled, Entity Resolved Knowledge Graph

Now we are going to put it all together as a geoERKG.  In notebook `03` we positioned resolved entities on a map.  Here we build an actual graph in memory via`networkx`, where resolved entities become nodes, the relationships Senzing published as match keys become edges, and the Placekey and coordinates ride along as properties on each node.

We will then use an LLM to ask questions and observe how the answers have changed from notebook `04` now that they are backed with the geoERKG.  Questions can be about any business or address in the data (not just the conference hotel) and the retrieval step works out where in the graph to look.

### How this notebook is assembled

Steps 1 to 5 run once. Steps 6 to 10 run again for every question you ask.

1. Connect to the Senzing resolver and to PostgreSQL.
2. Read the geocodes from `workshop.entity_geo`, keeping every address an entity has rather than just one.
3. Export all resolved entities from Senzing making each entity a node, with its name, sources, merged records, Placekey and coordinates attached as properties.
4. Add an edge for every relationship Senzing published, keeping the match key on the edge so we know why the two entities are connected.
5. Build the helper functions to do the following:
   
    - Distance from Placekey coordinates
    - Grouping by H3 cell
    - An anchor lookup that uses Senzing's search to identify a starting entity in the graph

<div style="height:0.2em"></div>
   
6. Send the question to a small, cheap LLM, which reads out how far to look and what anchor place to look around.
7. Resolve that place to an entity in the graph, defaulting to the conference hotel when the question names nowhere else.
8. Gather the neighborhood ofevery geocoded entity within the radius of that anchor, grouped by the building block it sits in.
9. Walk the entity resolution edges outward from those entities, including the links that reach outside the neighborhood entirely.
10. Flatten all of it into one block of text and hand it to a larger LLM to interpret.

In [ ]:
import collections, json, math, os, re, time

import anthropic
import grpc
from IPython.display import Markdown, display
import networkx as nx
import openai
import pandas as pd
from senzing import SzEngineFlags as F
from senzing_grpc import SzAbstractFactoryGrpc
from sqlalchemy import create_engine

channel = grpc.insecure_channel(
    f"{os.getenv('SENZING_GRPC_HOST', 'resolver')}:8261",
    options=[('grpc.max_receive_message_length', 200 * 1024 * 1024)])
grpc.channel_ready_future(channel).result(timeout=20)
engine = SzAbstractFactoryGrpc(channel).create_engine()
print("✅ resolver connected")

db = create_engine(
    f"postgresql+psycopg2://{os.getenv('POSTGRES_USER', 'postgres')}:"
    f"{os.getenv('POSTGRES_PASSWORD', 'workshop')}@"
    f"{os.getenv('POSTGRES_HOST', 'postgres')}:5432/{os.getenv('POSTGRES_DB', 'erkg')}")
print("✅ database connected")

HYATT = 2120          # the conference hotel - every question starts here

## Geocodes as node properties

We keep every address an entity has, not just one.  This is because, unlike in notebook 04 where the only thing we could do was deal with raw rows that each contained one address, we now have the ability to create a holistic view of an entity.  In that view, entities can have multiple addresses, which you will see makes for a more accurate RAG experience.  

In [ ]:
geo = pd.read_sql("""
    SELECT entity_id, street, city, placekey, latitude, longitude
    FROM workshop.entity_geo
    WHERE placekey IS NOT NULL AND street NOT LIKE 'PO BOX%%'
""", db)

addresses = collections.defaultdict(list)
for r in geo.itertuples():
    addresses[int(r.entity_id)].append(
        {'street': r.street, 'city': r.city, 'placekey': r.placekey,
         'lat': r.latitude, 'lon': r.longitude})

print(f"{len(addresses):,} entities have a placekey")

## Building the graph

Every resolved entity becomes a node, including the ones we could not geocode.  A node carries its name, address(es), the source(s) it came from, the records that merged into it, and its Placekey and coordinates when we have them.  The edges are the Senzing match keys, which are added in the next cell.

Note that this might take up to 1 minute to run.

In [ ]:
t0 = time.time()
G = nx.Graph()
pending_links = []

handle = engine.export_json_entity_report(F.SZ_EXPORT_DEFAULT_FLAGS)
try:
    while (line := engine.fetch_next(handle)):
        d = json.loads(line)
        e = d['RESOLVED_ENTITY']
        eid = e['ENTITY_ID']
        recs = e.get('RECORDS', [])
        a = addresses.get(eid, [])

        G.add_node(eid,
                   name=e.get('ENTITY_NAME') or f'Entity {eid}',
                   sources=sorted({r['DATA_SOURCE'] for r in recs}),
                   n_records=len(recs),
                   records=[f"{r['DATA_SOURCE']}:{r['RECORD_ID']}" for r in recs[:8]],
                   addresses=a,
                   placekey=a[0]['placekey'] if a else None,
                   street=a[0]['street'] if a else None,
                   city=a[0]['city'] if a else None,
                   lat=a[0]['lat'] if a else None,
                   lon=a[0]['lon'] if a else None)

        for r in d.get('RELATED_ENTITIES', []):
            pending_links.append((eid, r['ENTITY_ID'], r.get('MATCH_KEY', ''),
                                  bool(r.get('IS_AMBIGUOUS', 0))))
finally:
    engine.close_export_report(handle)

print(f"{G.number_of_nodes():,} entity nodes in {time.time() - t0:.0f}s")

## Edges from entity resolution

Senzing reports each relationship bidirectionally, so we add a pair only once.  

In [ ]:
for a, b, match_key, ambiguous in pending_links:
    if a < b and G.has_node(b):         
        G.add_edge(a, b, match_key=match_key, ambiguous=ambiguous)

print(f"{G.number_of_edges():,} entity-resolution edges")

## What we built

Note that not all nodes have a Placekey.  Only the conference neighborhood was geocoded even though the data we have is statewide for some of the sources.  This means that the graph is statewide while the geography is only local for this workshop.

In [ ]:
located = [n for n, a in G.nodes(data=True) if a['lat'] is not None]
keys = collections.Counter(d['match_key'] for _, _, d in G.edges(data=True))

print(f"entities          {G.number_of_nodes():,}")
print(f"  with a placekey {len(located):,}")
print(f"ER edges          {G.number_of_edges():,}")
print(f"  ambiguous       {sum(1 for _,_,d in G.edges(data=True) if d['ambiguous']):,}")
print(f"\nmatch keys: {keys.most_common(6)}")
print(f"\nanchor: {G.nodes[HYATT]['name']} at {G.nodes[HYATT]['street']}")

## Calculating distance from the Placekey

A placekey is written `what@where`. The `where` is a hexagon of known size, with an edge length of ~76 m and ~135 m across, so two entities sharing it are likely in the same or neighboring buildings.  Here we use the Haversine equation to get distance in meters, after converting the Placekey to lat/lon coordinates.

In [ ]:
def cell(placekey):
    """A placekey is what@where. The 'where' is the H3 hexagon."""
    return placekey.split('@')[-1]


def meters(a, b):
    x, y = G.nodes[a], G.nodes[b]
    la1, lo1, la2, lo2 = x['lat'], x['lon'], y['lat'], y['lon']
    return 6371000 * 2 * math.asin(math.sqrt(
        math.sin(math.radians(la2 - la1) / 2) ** 2 +
        math.cos(math.radians(la1)) * math.cos(math.radians(la2)) *
        math.sin(math.radians(lo2 - lo1) / 2) ** 2))


# A sanity check on meters(): two businesses sharing a placekey must read 0,
# because they share a cell centroid. One in a different cell must not.
for other, note in ((1259, 'same placekey as the hotel'),
                    (2783, "MAX'S, a few hundred meters away")):
    print(f"{G.nodes[HYATT]['name'][:28]} -> {G.nodes[other]['name'][:28]}: "
          f"{meters(HYATT, other):>5.0f} m   ({note})")

## Gathering the neighborhood

This is our retrieval step, which is every entity within a given distance of an anchor, grouped by the H3 cell its placekey falls in.  The anchor is whatever place the question is about, worked out a few cells below, and it defaults to the conference hotel when the question names nowhere else.

Grouping is on the `where` half of the placekey, not the whole thing, so suites and separate addresses that share a hexagon collapse into one group.

In [ ]:
def neighborhood(anchor=HYATT, radius_m=600, max_places=25):
    """Every located entity within radius_m of the anchor, grouped by H3 cell."""
    anchor_cell = cell(G.nodes[anchor]['placekey'])
    near = sorted(((n, meters(anchor, n)) for n in located if n != anchor),
                  key=lambda t: t[1])

    by_place = collections.OrderedDict()
    for n, d in near:
        c = cell(G.nodes[n]['placekey'])
        by_place.setdefault(c, {'d': d, 'ents': []})
        if d <= radius_m or c == anchor_cell:
            by_place[c]['ents'].append(n)

    places = {c: v for c, v in by_place.items() if v['ents']}
    places = dict(list(places.items())[:max_places])
    members = {anchor} | {n for v in places.values() for n in v['ents']}
    return places, members


places, members = neighborhood()
print(f"{len(places)} locations, {len(members)} entities")

## Turning the neighborhood into context

One function builds everything the model sees: who is at the anchor location, what is nearby and how far, which entities were linked by the ER process and on what evidence, which links reach outside the neighborhood, and which addresses look unreliable.  There is no separate code path per question, which is what lets one function answer a wide range of them.

Note what arrives here that distance alone could never find.  Businesses several kilometers away, and even some with no address at all, are pulled into the context because an entity resolution edge points at them.  That is the graph doing work no vector search or map query could do.

In [ ]:
def describe(anchor=HYATT, radius_m=600, max_places=200):
    places, members = neighborhood(anchor, radius_m, max_places)
    A = G.nodes[anchor]

    out = [f"STARTING POINT: {A['name']} (entity {anchor})",
           f"  {A['street']}, {A['city']} | placekey {A['placekey']} "
           f"| sources: {', '.join(A['sources'])}",
           f"  source records merged into this one entity: {', '.join(A['records'])}",
           "",
           "SURROUNDING LOCATIONS - meters from the starting point, computed from",
           "Placekey coordinates.",
           ""]

    for c, v in places.items():
        streets = sorted({a['street'] for n in v['ents'] for a in G.nodes[n]['addresses']
                          if cell(a['placekey']) == c})
        out.append(f"{v['d']:.0f} m - {streets[0]}  [cell {c}]")
        for n in v['ents'][:12]:
            a = G.nodes[n]
            out.append(f"    {n} {a['name']} | {', '.join(a['sources'])} "
                       f"| {a['n_records']} record(s)")
        if len(v['ents']) > 12:
            out.append(f"    ... and {len(v['ents']) - 12} more")

    out += ["", "ENTITY RESOLUTION LINKS. Senzing decided these are related; the key",
            "says why. +ADDRESS alone means only a shared address. +NAME or +PHONE",
            "means the match would survive the business moving premises."]

    inner, outer = [], []
    for m in sorted(members):
        for nb in G.neighbors(m):
            if nb < m and nb in members:
                continue
            (inner if nb in members else outer).append((m, nb, G.edges[m, nb]))

    # Most links are +ADDRESS, which is near-tautological with sharing a placekey.
    # Sort the address-independent ones first so truncation cannot discard them.
    def interest(t):
        d = t[2]
        return (0 if d['ambiguous'] else 1,
                0 if ('+NAME' in d['match_key'] or '+PHONE' in d['match_key']) else 1)

    inner.sort(key=interest)
    outer.sort(key=interest)

    def fmt(m, nb, d):
        return (f"  {m} {G.nodes[m]['name'][:32]} <-> {nb} {G.nodes[nb]['name'][:32]}"
                f"  {d['match_key']}{' (AMBIGUOUS)' if d['ambiguous'] else ''}")

    out.append("  -- between businesses in this neighborhood:")
    out += [fmt(m, nb, d) for m, nb, d in inner[:50]]

    out.append("  -- reaching OUTSIDE this neighborhood:")
    for m, nb, d in outer[:30]:
        b = G.nodes[nb]
        if b['lat'] is not None:
            out.append(fmt(m, nb, d) +
                       f"  [{b['street']}, {meters(m, nb):.0f} m from here]")
        else:
            out.append(fmt(m, nb, d) + "  [no address on file]")

    out += ["", "DATA QUALITY. A Placekey is what@where. A long 'what' beginning with 0",
            "means the address could not be matched, so its coordinates are unreliable."]
    for n in sorted(members):
        a = G.nodes[n]
        cells = {cell(x['placekey']) for x in a['addresses']}
        if len(cells) > 1:
            out.append(f"  entity {n} {a['name']} has {len(a['addresses'])} addresses "
                       f"in {len(cells)} different cells:")
            out += [f"      {x['street']} -> {x['placekey']}" for x in a['addresses']]

    return "\n".join(out)

### A quick check that we are getting what we expect in the nearby neighborhood

With no anchor passed, `describe()` falls back to the conference hotel.

In [ ]:
ctx = describe()
print(f"{len(ctx):,} characters (~{len(ctx)//4:,} tokens)\n")
print(ctx[:1400])

## When there is no geography

Only 2,376 of the 31,339 entities were geocoded, because only the conference neighborhood was sent to Placekey yet our data contains records and entities that are outside of this region.  So a perfectly reasonable question can land on an entity that has no coordinates at all, and there is no neighborhood to gather around it.

Rather than refuse to answer, we fall back to what entity resolution knows on its own: every name variant it merged, the addresses and phone numbers on file, the records it was built from, and the other entities it is related to along with the match key for each.

This is worth seeing in its own right.  `DESILVA GATES CONSTRUCTION LLC` is one entity built from 13 licenses across two agencies, known by ten different names, and linked to five other companies.  None of that needs a single coordinate.  The entity resolution layer and the geographic layer are independent, and each is useful without the other.

In [ ]:
def describe_entity(eid, max_values=8, max_links=15):
    """Context for an entity with no geocode, built from entity resolution alone.

    Only 2,376 of 31,339 entities were geocoded, so a question can easily land on
    one with no coordinates. Rather than refuse, we fall back to what resolution
    knows: every name it merged, the addresses and phones on file, the records it
    was built from, and the entities it is related to."""
    d = json.loads(engine.get_entity_by_entity_id(eid, flags=F.SZ_ENTITY_DEFAULT_FLAGS))
    e = d['RESOLVED_ENTITY']
    feats = e.get('FEATURES', {})
    recs = e.get('RECORDS', [])
    sources = sorted({r['DATA_SOURCE'] for r in recs})

    out = [f"ENTITY {eid}: {e.get('ENTITY_NAME', '')}",
           "",
           "NO COORDINATES. This entity was never geocoded, so there are no distances",
           "in this context and nothing about what is nearby. Everything below comes",
           "from entity resolution alone.",
           "",
           f"Built from {len(recs)} source record(s) across {len(sources)} agency "
           f"source(s): {', '.join(sources)}."]

    for fam, label in (('NAME', 'NAMES this entity is known by'),
                       ('ADDRESS', 'ADDRESSES on file'),
                       ('PHONE', 'PHONE numbers')):
        vals = feats.get(fam, [])
        if vals:
            out.append(f"\n{label} ({len(vals)}):")
            out += [f"    {v['FEAT_DESC']}" for v in vals[:max_values]]
            if len(vals) > max_values:
                out.append(f"    ... and {len(vals) - max_values} more")

    out.append(f"\nSOURCE RECORDS ({len(recs)}):")
    out += [f"    {r['DATA_SOURCE']}:{r['RECORD_ID']}" for r in recs[:max_values * 2]]
    if len(recs) > max_values * 2:
        out.append(f"    ... and {len(recs) - max_values * 2} more")

    rel = d.get('RELATED_ENTITIES', [])
    out.append("\nENTITY RESOLUTION LINKS. Senzing decided these are related but kept")
    out.append("them separate; the key says why. +ADDRESS alone means only a shared")
    out.append("address, while +NAME or +PHONE survives a change of premises.")
    if not rel:
        out.append("    (none)")
    for r in rel[:max_links]:
        line = (f"    {r['ENTITY_ID']:>7} {r.get('ENTITY_NAME', '')[:40]:<40} "
                f"{r.get('MATCH_KEY', '')}  {r.get('MATCH_LEVEL_CODE', '')}")
        nb = r['ENTITY_ID']
        if G.has_node(nb) and G.nodes[nb]['street']:
            line += f"  [at {G.nodes[nb]['street']}]"
        out.append(line)
    if len(rel) > max_links:
        out.append(f"    ... and {len(rel) - max_links} more")

    return "\n".join(out)


print(describe_entity(1354)[:700])

## Finding where in the graph to start

A question may be about any address or business in the data, so something has to turn "1249 Broadway" or "Max's of Burlingame" into an entity to measure from.

We use Senzing's `search_by_attributes` for this, the same call we used in notebook `02` and the same matcher that built the graph.  Given `ADDR_FULL` it parses a whole address string itself, so we need no address parsing of our own, and it returns a match level that lets us prefer the strongest candidate rather than the first one.

Two limits are worth knowing.  Address search wants a city, so we supply Burlingame when the question does not.  Also, search needs a near-complete name, which is why a substring sweep over the graph's node names is the last resort.  Searching and resolving are different operations, and search is the stricter of the two.

In [ ]:
DEFAULT_CITY = "BURLINGAME"

# Senzing's own search ranks its hits, so prefer the strongest match rather than
# the first one. RESOLVED means name and address both agreed.
MATCH_RANK = {'RESOLVED': 0, 'POSSIBLY_SAME': 1, 'POSSIBLY_RELATED': 2,
              'NAME_ONLY': 3, 'DISCLOSED': 4}


def _search(attrs):
    """search_by_attributes takes the same attributes you would load as a record
    and says which existing entities it would match, storing nothing."""
    r = json.loads(engine.search_by_attributes(json.dumps(attrs)))
    hits = []
    for h in r.get('RESOLVED_ENTITIES', []):
        e = h['ENTITY']['RESOLVED_ENTITY']
        mi = h.get('MATCH_INFO', {})
        hits.append((MATCH_RANK.get(mi.get('MATCH_LEVEL_CODE', ''), 9),
                     e['ENTITY_ID'], mi.get('MATCH_KEY', '')))
    hits.sort()
    return hits


def resolve_anchor(text):
    """Turn a place or business name from a question into an entity to anchor on.

    Senzing does the work: ADDR_FULL parses a whole address string, so we need no
    regex of our own. It wants a city for addresses and a near-complete name for
    names, so a substring sweep over the graph is the last resort - searching and
    resolving are different operations, and search is the stricter of the two."""
    tries = []
    if re.match(r'\s*\d', text):            # starts with a number: an address
        full = text if ',' in text else f"{text}, {DEFAULT_CITY}, CA"
        tries.append(({'ADDR_FULL': full}, 'Senzing address search'))
    tries.append(({'NAME_ORG': text, 'ADDR_CITY': DEFAULT_CITY}, 'Senzing name search'))
    tries.append(({'NAME_ORG': text}, 'Senzing name search, no city'))

    for attrs, how in tries:
        for _, eid, key in _search(attrs):
            if eid in located:              # must be geocoded to measure from
                return eid, f"{how} ({key})"

    flat = re.sub(r'[^A-Z0-9]', '', text.upper())
    if len(flat) >= 4:
        matches = [n for n in G.nodes
                   if flat in re.sub(r'[^A-Z0-9]', '', G.nodes[n]['name'].upper())]
        for n in sorted(matches, key=lambda n: (n not in located, n)):
            return n, ('substring over graph names' if n in located
                       else 'substring, but NOT geocoded')

    for _, eid, key in _search({'NAME_ORG': text}):
        return eid, f"found but NOT geocoded ({key})"
    return None, 'no match'


for t in ["1249 Broadway", "Max's of Burlingame", "Topgolf",
          "Grand Harbor Seafood", "DESILVAGATES CONSTRUCTION", "Nowhere Diner"]:
    n, how = resolve_anchor(t)
    print(f"  {t[:28]:<28} -> "
          + (f"{n:<7} {G.nodes[n]['name'][:30]:<30} [{how}]" if n else how))


## Letting the model plan the retrieval

The question usually says both what it is about and how far to look, so rather than hard coding either, we ask a small cheap LLM to read them out of the question.  It returns a radius in meters and, if the question names one, the business or address to center on.  We give it instructions on what is meant by phrases like "walking distance," "blocks," or "a 10 minute walk."

This is the only call that sees the question by itself, with no data attached.  Everything it returns is then used by ordinary Python to query the graph.

**Important Note**

Remember that distance is being measured between Placekey coordinates and not along a map of city blocks, so the actual walking distance is likely longer.  Because those coordinates are hexagon centers, anything under about 150 m is approximate.  Comparisons between distances stay reliable even where the absolute numbers are rough.


In [ ]:
anthropic_client = anthropic.Anthropic() if os.getenv("ANTHROPIC_API_KEY") else None
openai_client    = openai.OpenAI()       if os.getenv("OPENAI_API_KEY")    else None

# Two jobs, two sizes. The planner only reads a distance and a target out of a
# sentence, so a small fast model is plenty. The answer model reasons over the
# whole neighborhood, so it gets the largest one.
ANSWER_MODEL  = {"anthropic": "claude-opus-5",    "openai": "gpt-5.4-nano"}
PLANNER_MODEL = {"anthropic": "claude-haiku-5-5", "openai": "gpt-5.4-nano"}

# Pick whichever key is present, so nobody has to edit function signatures.
# Set LLM_PROVIDER in .env to force one when both keys are set.
PROVIDER = os.getenv("LLM_PROVIDER") or ("anthropic" if anthropic_client else "openai")
if PROVIDER == "anthropic" and not anthropic_client:
    raise RuntimeError("LLM_PROVIDER=anthropic but ANTHROPIC_API_KEY is not set")
if PROVIDER == "openai" and not openai_client:
    raise RuntimeError("LLM_PROVIDER=openai but OPENAI_API_KEY is not set")

print("Anthropic key present:", anthropic_client is not None)
print("OpenAI key present:   ", openai_client is not None)
print(f"Using: {PROVIDER}  answer={ANSWER_MODEL[PROVIDER]}  planner={PLANNER_MODEL[PROVIDER]}")

In [ ]:
PLANNER_PROMPT = """You read two things out of a question about businesses near a hotel,
    and reply with ONLY a JSON object, nothing else.

    {"radius": <integer meters, or null>, "target": "<place or business, or null>"}

    radius - how far to search:
      '1000m' -> 1000      '2 km' -> 2000       'half a mile' -> 805
      'a mile' -> 1609     '3 blocks' -> 360    '10 minute walk' -> 800
      (a block is 120 m; walking is 80 m per minute)
      'the same building', 'the same address', 'shares this address' -> 1
      'next door', 'two doors down', 'immediately outside' -> 250
      'near', 'nearby', 'close by', 'around here' -> 600
      'walk', 'walking distance', 'on foot' -> 800
      'this neighborhood', 'the area', 'the wider area' -> 1500
      'anywhere', 'in Burlingame', 'the whole town' -> 3000
      null if the question says nothing about distance or place at all.

    target - the address or business name the question is ABOUT, copied from the
    question. null when the question is about where the user is standing, or
    about the data as a whole.

    IMPORTANT: if you are asked about "the hotel" or something about being in the
    same building or if the target is not clearly defined, it means the 
    Hyatt Regency San Francisco Airport as the target.

    Examples:
      Q: What else operates out of the same building?
         {"radius": 1, "target": null}
      Q: Anywhere I could walk to for a drink?
         {"radius": 800, "target": null}
      Q: At 1249 Broadway there seem to be several salons. One business or several?
         {"radius": 1, "target": "1249 Broadway"}
      Q: Tell me about Max's of Burlingame
         {"radius": 1, "target": "Max's of Burlingame"}
      Q: What is within 500 m of Topgolf?
         {"radius": 500, "target": "Topgolf"}
      Q: Which links is the system unsure about?
         {"radius": null, "target": null}
    Reply with the JSON object only."""


def plan(question, provider=None):
    """Ask the small model what to retrieve: how far, and around what.
    Returns (radius_m or None, target or None). No fallback values here."""
    provider = provider or PROVIDER
    if provider == "anthropic":
        r = anthropic_client.messages.create(
            model=PLANNER_MODEL["anthropic"], max_tokens=2000,
            system=PLANNER_PROMPT,
            messages=[{"role": "user", "content": question}])
        text = "".join(b.text for b in r.content if b.type == "text").strip()
    else:
        r = openai_client.chat.completions.create(
            model=PLANNER_MODEL["openai"], max_completion_tokens=2000,
            messages=[{"role": "system", "content": PLANNER_PROMPT},
                      {"role": "user", "content": question}])
        text = (r.choices[0].message.content or "").strip()

    m = re.search(r'\{.*\}', text, re.S)
    if not m:
        return None, None
    try:
        d = json.loads(m.group())
    except ValueError:
        return None, None
    radius = d.get('radius')
    radius = max(1, min(int(radius), 5000)) if isinstance(radius, (int, float)) else None
    return radius, (d.get('target') or None)


for q in ["What else operates out of the same building?",
          "At 1249 Broadway there seem to be several salons. One business or several?",
          "Tell me about Max's of Burlingame",
          "What is within 500 m of Topgolf?",
          "Which links is the system unsure about?"]:
    print(f"  {str(plan(q)):<34} {q[:48]}")


## The system prompt

Now we describe the complete process for the LLM, including the geographic information as well as the output of Senzing.

In [ ]:
SYSTEM_PROMPT = """You answer questions about California businesses using a knowledge 
    graph built from public records. The records were entity-resolved by Senzing - 
    one entity is one real business, built from several source records - and 
    geocoded with Placekey. The context is centered on whatever place the question 
    asked about, which may or may not be the conference hotel. 
    
    DISTANCES ARE COMPUTED, NOT GUESSED: they are straight-line meters from 
    Placekey coordinates, so you may quote them and compare them. Treat 400 m as 
    a short walk, 800 m as about ten minutes. 
    
    Distances are measured between the centers of Placekey cells, which are 
    hexagons about 135 m across. So two businesses reading 0 m apart share a 
    cell and are on the same block, which often but not always means the same 
    building. Any distance under about 150 m is approximate, and you should say 
    so rather than implying precision you do not have. Comparisons between 
    distances are reliable even where the absolute figures are rough.
    
    SOMETIMES THERE IS NO GEOGRAPHY. Only part of the data was geocoded, so a 
    context may say NO COORDINATES. That is normal and not an error. Answer fully 
    from the resolution evidence you were given - the merged records, the name 
    variants, the addresses on file and the links - and mention the absence of 
    coordinates only if the question actually asked about distance or proximity. 
    
    ABC-RETAIL and ABC-NONRETAIL mean a liquor license; no other source implies 
    alcohol. 
    
    Links marked +ADDRESS, +NAME or +PHONE were made by entity resolution - say 
    which key supports a claim. AMBIGUOUS means Senzing is unsure. 
    Answer only from the context, cite entity IDs, and say plainly when the 
    context does not support an answer.
"""

## The full QA bot, backed with the geoERKG

This is the function that handles the complete question answering.  It asks the small LLM for a radius and a target, resolves that target to an entity in the graph, walks the graph outward from there, and hands the result to the larger LLM to interpret.

It prints which entity it anchored on and how it found it.  When an answer looks wrong, that is the first thing to check, because a correct answer about the wrong part of the graph reads exactly like a correct answer.

In [ ]:
def ask(question, provider=None, max_tokens=8000):
    """Ask about anywhere in the data, not just the hotel. The planner reads the
    target and the distance out of the question; the anchor defaults to the hotel
    when the question names no other place. An anchor with no coordinates falls
    back to an entity resolution only answer."""
    provider = provider or PROVIDER
    radius_m, target = plan(question, provider)

    geo_context = True
    if target:
        anchor, how = resolve_anchor(target)
        if anchor is None:
            print(f"❌ Could not find '{target}' in the data. Try the exact business\n"
                  "   name, or a street address with its number.")
            return
        anchor_note = f"'{target}' -> entity {anchor} {G.nodes[anchor]['name']} [{how}]"
        geo_context = G.nodes[anchor]['lat'] is not None
    else:
        anchor, anchor_note = HYATT, f"no target named, using the hotel (entity {HYATT})"

    if geo_context:
        # No distance stated: a named target means just its location, no target
        # means the question is about the dataset, so search everything.
        if radius_m is None:
            radius_m = 1 if target else 5000
            scope = ("no distance given, so just this location" if target
                     else "no distance given, so searching every geocoded business")
        else:
            scope = "from the question"
        context = describe(anchor=anchor, radius_m=radius_m)
        detail = f"Searching within {radius_m} m ({scope})"
    else:
        # Not geocoded, so there is no neighborhood to gather. Answer from the
        # resolution evidence instead of refusing.
        context = describe_entity(anchor)
        detail = "No coordinates for this entity, so answering from resolution only"

    print(f"Question: {question}")
    print(f"Anchor:   {anchor_note}")
    print(detail)
    print(f"Context:  {len(context):,} chars (~{len(context)//4:,} tokens)")

    msg = f"Knowledge graph context:\n{context}\n\nQuestion: {question}"

    try:
        if provider == "anthropic":
            r = anthropic_client.messages.create(
                model=ANSWER_MODEL["anthropic"], max_tokens=max_tokens,
                system=SYSTEM_PROMPT, messages=[{"role": "user", "content": msg}])
            answer = "".join(b.text for b in r.content if b.type == "text")
            if not answer:
                print(f"⚠️  no text returned. stop_reason={r.stop_reason}, "
                      f"blocks={[b.type for b in r.content]}, "
                      f"output_tokens={r.usage.output_tokens}")
                return
        else:
            r = openai_client.chat.completions.create(
                model=ANSWER_MODEL["openai"], max_completion_tokens=max_tokens,
                messages=[{"role": "system", "content": SYSTEM_PROMPT},
                          {"role": "user", "content": msg}])
            answer = r.choices[0].message.content
            if not answer:
                print(f"⚠️  no text returned. finish_reason={r.choices[0].finish_reason}")
                return
    except openai.RateLimitError as e:
        if getattr(e, "code", "") == "insufficient_quota":
            print("❌ Your OpenAI account has no credit. This is a billing issue, not\n"
                  "   a rate limit, despite the exception name.")
        else:
            print(f"❌ OpenAI rate limit hit. This question sent ~{len(context)//4:,} tokens.")
        return
    except anthropic.RateLimitError:
        print("❌ Anthropic rate limit hit. Wait a moment and re-run.")
        return

    display(Markdown(answer))

## Questions

Experiment with this section with questions of your own.  Here are some good ones to try:

- I am staying at this hotel.  What else operates out of the same building?
- Where's the nearest place to the Hyatt Regency San Francisco Airport I could get a drink, and how far is the walk?
- Which businesses around the Hyatt Regency San Francisco Airport are connected to each other, and what evidence connects them?
- Are any of the businesses near the Hyatt Regency San Francisco Airport part of something larger elsewhere in California?
- Is anything in this data wrong or suspicious?
- Find a bar not in a hotel that is within 1000 m.
- Tell me about Top Golf and can I get a drink there?

In [ ]:
ask("What businesses are near the hotel?")

In [ ]:
ask("Tell me about Max's of Burlingame")

In [ ]:
ask("How many separate licenses does DeSilva Gates Construction hold, "
        "and is DESILVAGATES CONSTRUCTION the same company?")

In [ ]:
ask("At 1249 Broadway there seem to be several salons with similar names. One business or several?")

In [ ]:
ask("Where's the nearest place to the hotel I could get a drink, and how far is the walk?")

In [ ]:
ask("Where can I get a drink nearby that isn't in or part of a hotel?")